In [1]:
import pandas as pd
import numpy as np
import gc

In [2]:
DATA_DIR = "datasets"

In [3]:
def downcast(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    return df

In [4]:
def aggregate_numeric(df, group_col, prefix, stats=('mean', 'max', 'min', 'sum')):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if group_col in numeric_cols:
        numeric_cols.remove(group_col)
 
    agg = df.groupby(group_col)[numeric_cols].agg(list(stats))
    agg.columns = [f"{prefix}_{col}_{stat}" for col, stat in agg.columns]
    agg = downcast(agg.reset_index())
    return agg

In [5]:
def aggregate_categorical(df, group_col, prefix):
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    if not cat_cols:
        return df[[group_col]].drop_duplicates()
 
    result = df[[group_col]].drop_duplicates().set_index(group_col)
 
    for col in cat_cols:
        # crosstab of counts is far lighter than get_dummies + merge
        counts = pd.crosstab(df[group_col], df[col])
        counts = counts.div(counts.sum(axis=1), axis=0)  # convert to fraction
        counts.columns = [f"{prefix}_{col}_{c}_mean" for c in counts.columns]
        counts = downcast(counts)
        result = result.join(counts, how='left')
        del counts
        gc.collect()
 
    result.reset_index(inplace=True)
    return result

In [6]:
def aggregate_table(df, group_col, prefix):
    num_agg = aggregate_numeric(df, group_col, prefix)
    cat_agg = aggregate_categorical(df, group_col, prefix)
    out = num_agg.merge(cat_agg, on=group_col, how='left')
    del num_agg, cat_agg
    gc.collect()
    return out

In [7]:
def main():
    print("Loading raw CSVs...")
    app_train = pd.read_csv(f"{DATA_DIR}/application_train.csv")
    app_test = pd.read_csv(f"{DATA_DIR}/application_test.csv")
    bureau = pd.read_csv(f"{DATA_DIR}/bureau.csv")
    bureau_balance = pd.read_csv(f"{DATA_DIR}/bureau_balance.csv")
    prev_app = pd.read_csv(f"{DATA_DIR}/previous_application.csv")
    pos_cash = pd.read_csv(f"{DATA_DIR}/POS_CASH_balance.csv")
    credit_card = pd.read_csv(f"{DATA_DIR}/credit_card_balance.csv")
    installments = pd.read_csv(f"{DATA_DIR}/installments_payments.csv")
 
    # -----------------------------------------------------------------
    # STEP 1: bureau_balance (rows per SK_ID_BUREAU) -> squash -> merge
    # into bureau -> squash bureau (rows per SK_ID_CURR) -> 1 feature table
    # -----------------------------------------------------------------
    print("Step 1: Aggregating bureau_balance -> bureau...")
    bb_agg = aggregate_table(bureau_balance, 'SK_ID_BUREAU', 'BB')
    del bureau_balance
    gc.collect()
    bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
    del bb_agg
    gc.collect()
    bureau_features = aggregate_table(bureau, 'SK_ID_CURR', 'BUREAU')
    del bureau
    gc.collect()
    print(f"  -> bureau_features shape: {bureau_features.shape}")
 
    # -----------------------------------------------------------------
    # STEP 2: pos_cash, credit_card, installments (rows per SK_ID_PREV)
    # -> squash each -> merge into previous_application
    # -> squash previous_application (rows per SK_ID_CURR) -> 1 feature table
    # -----------------------------------------------------------------
    print("Step 2: Aggregating POS_CASH, credit_card, installments -> previous_application...")
    print("  -> aggregating POS_CASH_balance...")
    pos_agg = aggregate_table(pos_cash, 'SK_ID_PREV', 'POS')
    del pos_cash
    gc.collect()
    prev_app = prev_app.merge(pos_agg, on='SK_ID_PREV', how='left')
    del pos_agg
    gc.collect()
 
    print("  -> aggregating credit_card_balance...")
    cc_agg = aggregate_table(credit_card, 'SK_ID_PREV', 'CC')
    del credit_card
    gc.collect()
    prev_app = prev_app.merge(cc_agg, on='SK_ID_PREV', how='left')
    del cc_agg
    gc.collect()
 
    print("  -> aggregating installments_payments...")
    inst_agg = aggregate_table(installments, 'SK_ID_PREV', 'INST')
    del installments
    gc.collect()
    prev_app = prev_app.merge(inst_agg, on='SK_ID_PREV', how='left')
    del inst_agg
    gc.collect()
 
    prev_app_features = aggregate_table(prev_app, 'SK_ID_CURR', 'PREV')
    del prev_app
    gc.collect()
    print(f"  -> prev_app_features shape: {prev_app_features.shape}")
 
    # -----------------------------------------------------------------
    # STEP 3: Merge everything onto application_train / application_test
    # -----------------------------------------------------------------
    print("Step 3: Merging into application_train and application_test...")
    train_merged = app_train.merge(bureau_features, on='SK_ID_CURR', how='left')
    train_merged = train_merged.merge(prev_app_features, on='SK_ID_CURR', how='left')
 
    test_merged = app_test.merge(bureau_features, on='SK_ID_CURR', how='left')
    test_merged = test_merged.merge(prev_app_features, on='SK_ID_CURR', how='left')
 
    print(f"  -> train_merged shape: {train_merged.shape}")
    print(f"  -> test_merged shape: {test_merged.shape}")
 
    # -----------------------------------------------------------------
    # STEP 4: Save final files so you never have to re-run this again
    # -----------------------------------------------------------------
    train_merged.to_csv(f"{DATA_DIR}/train_merged.csv", index=False)
    test_merged.to_csv(f"{DATA_DIR}/test_merged.csv", index=False)
    print("Done. Saved train_merged.csv and test_merged.csv")

In [8]:
if __name__ == "__main__":
    main()

Loading raw CSVs...
Step 1: Aggregating bureau_balance -> bureau...
  -> bureau_features shape: (305811, 124)
Step 2: Aggregating POS_CASH, credit_card, installments -> previous_application...
  -> aggregating POS_CASH_balance...
  -> aggregating credit_card_balance...
  -> aggregating installments_payments...
  -> prev_app_features shape: (338857, 832)
Step 3: Merging into application_train and application_test...
  -> train_merged shape: (307511, 1076)
  -> test_merged shape: (48744, 1075)
Done. Saved train_merged.csv and test_merged.csv
